# Reporting Views — Healthcare Analytics
Creates SQL views for Power BI and ad-hoc reporting: executive summary, bed occupancy, readmission hotspots, department scorecards.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_executive_summary AS
SELECT
    admission_date_only            AS report_date,
    SUM(admissions)                AS total_admissions,
    SUM(readmissions)              AS total_readmissions,
    ROUND(SUM(readmissions) / SUM(admissions) * 100, 2) AS overall_readmission_rate,
    ROUND(AVG(avg_los), 2)         AS overall_avg_los,
    SUM(high_risk_count)           AS total_high_risk,
    COUNT(DISTINCT department)     AS departments_active
FROM gold_department_daily_summary
GROUP BY admission_date_only
ORDER BY admission_date_only
""")
print('Created vw_executive_summary')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_bed_occupancy AS
SELECT
    admission_date_only,
    department,
    admissions          AS daily_admissions,
    avg_los,
    ROUND(admissions * avg_los, 0) AS estimated_patient_days,
    readmission_rate,
    dept_performance
FROM gold_department_daily_summary
""")
print('Created vw_bed_occupancy')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_readmission_hotspots AS
SELECT
    primary_dx_code,
    insurance_type,
    age_group,
    total_admissions,
    readmissions,
    readmission_rate,
    avg_los
FROM gold_readmission_analysis
WHERE readmission_rate > 10
ORDER BY readmission_rate DESC
""")
print('Created vw_readmission_hotspots')

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_department_scorecard AS
SELECT
    department,
    total_admissions,
    readmission_rate,
    avg_los,
    daily_admissions_avg,
    high_risk_count,
    performance_rank,
    CASE
        WHEN readmission_rate <= 5  THEN 'Excellent'
        WHEN readmission_rate <= 10 THEN 'Good'
        WHEN readmission_rate <= 15 THEN 'Needs Improvement'
        ELSE 'Critical'
    END AS performance_label
FROM gold_department_scorecard
ORDER BY performance_rank
""")
print('Created vw_department_scorecard')